<a href="https://colab.research.google.com/github/malyadri6761/RAG/blob/master/rag_05.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---

**Multi-PDF RAG**

---

```
  1. Uploading multiple PDFs
  2. Extract text page by page
  3. chunking with overlap
  4. Sentence Transformer embeddings
  5. FAISS vector index
  6. User question -> Top-k Retrieval
  7. Context from relevant PDFs
  8. Groq LLM -> Answer + sources

```

---

**Install and Import Libraries**

---

In [2]:
!pip install -q sentence-transformers faiss-cpu groq pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 9.0 MB/s eta 0:00:00


In [4]:
from sentence_transformers import SentenceTransformer
from pypdf import PdfReader
from groq import Groq
from google.colab import files,userdata


import numpy as np
import faiss

---

**Configure Groq**

---

In [5]:
groq_api_key = userdata.get("GROQ_API_KEY")
model = Groq(api_key = groq_api_key)

---

**Uploading Multiple Files**

----

In [6]:
path = ["/content/drive/MyDrive/A-Textbook-of-EngineeringP-hysics-by-M-N-Avadhanulu-and-PG-Kshirsagar.pdf",
        "/content/drive/MyDrive/Copy of Core Java - Black Book by Nageswara Rao.pdf",
        "/content/drive/MyDrive/Computer Organization And Architecture .pdf"]



In [7]:
documents = []
for filename in path:
  if not filename.lower().endswith(".pdf"):
    print(f"Skipping non-pdf file : {filename}")
    continue


  reader = PdfReader(filename)

  for page_number,page in enumerate(reader.pages):
    text = page.extract_text()

    if text.strip():
      documents.append({
          'text' : text,
          'source': filename,
          'page': page_number
      })

print("Total extracted pages : ",len(documents))
print("PDF files : ",len(set(doc['source'] for doc in documents)))


Total extracted pages :  1807
PDF files :  3


---

**Split into Chunks**

---

In [8]:
def create_chunks(text,chunk_size=100,overlap=20):
  words = text.split()
  chunks = []

  start = 0

  while start < len(words):
    end = start + chunk_size;
    chunk = " ".join(words[start:end])

    if chunk.strip():
      chunks.append(chunk)

    start += chunk_size - overlap
  return chunks

---

**Create chunks from every PDF**

---

In [15]:
all_chunks = []

for document in documents:
  chunks = create_chunks(
      document['text'],
  )
  for chunk in chunks:
    all_chunks.append({
        'text' : chunk,
        'source' : document['source'],
        'page' : document['page']


    })
print("Total chunks : ",len(all_chunks))
for i in range(5):
  print(f"chunk : {all_chunks[i]['text']}")
  print(f"source : {all_chunks[i]['source']}")
  print(f"page : {all_chunks[i]['page']}")
  print("-"*50)


Total chunks :  8996
chunk : A TEXTBOOK OF ENGINEERING PHYSICS [For the Students of B.E., B.Tech., B.Arch., B.Sc., (Engg.)] Dr. M.N. Avadhanulu M.Sc.,Ph.D. Ex-Principal, Om College of Engineering of Wardha Former Professor and Head, Department of Physics Kavikulguru Institute of Technology & Science Ramtek-441 106, Dist. Nagpur (M.S.) and Dr. P.G. Kshirsagar M.Sc., Ph.D. Formerly Head of the Department of Applied Physics, Visvesvaraya National Institute of Technology NAGPUR [Revised Edition] S. CHAND & COMPANY PVT. LTD. (AN ISO 9001 : 2008 COMPANY) RAM NAGAR, NEW DELHI-110 055
source : /content/drive/MyDrive/A-Textbook-of-EngineeringP-hysics-by-M-N-Avadhanulu-and-PG-Kshirsagar.pdf
page : 1
--------------------------------------------------
chunk : DELHI-110 055
source : /content/drive/MyDrive/A-Textbook-of-EngineeringP-hysics-by-M-N-Avadhanulu-and-PG-Kshirsagar.pdf
page : 1
--------------------------------------------------
chunk : S. CHAND & COMPANY PVT. LTD. (An ISO 9001 : 2008 Compa

In [16]:
## Another way
for i,chunk in enumerate(all_chunks[:5]):
  print(f"chunk : {i}")
  print(f"source : {chunk['source']}")
  print(f"page : {chunk['page']}")
  print(f"tex t: {chunk["text"]}")
  print("-"*50)

chunk : 0
source : /content/drive/MyDrive/A-Textbook-of-EngineeringP-hysics-by-M-N-Avadhanulu-and-PG-Kshirsagar.pdf
page : 1
tex t: A TEXTBOOK OF ENGINEERING PHYSICS [For the Students of B.E., B.Tech., B.Arch., B.Sc., (Engg.)] Dr. M.N. Avadhanulu M.Sc.,Ph.D. Ex-Principal, Om College of Engineering of Wardha Former Professor and Head, Department of Physics Kavikulguru Institute of Technology & Science Ramtek-441 106, Dist. Nagpur (M.S.) and Dr. P.G. Kshirsagar M.Sc., Ph.D. Formerly Head of the Department of Applied Physics, Visvesvaraya National Institute of Technology NAGPUR [Revised Edition] S. CHAND & COMPANY PVT. LTD. (AN ISO 9001 : 2008 COMPANY) RAM NAGAR, NEW DELHI-110 055
--------------------------------------------------
chunk : 1
source : /content/drive/MyDrive/A-Textbook-of-EngineeringP-hysics-by-M-N-Avadhanulu-and-PG-Kshirsagar.pdf
page : 1
tex t: DELHI-110 055
--------------------------------------------------
chunk : 2
source : /content/drive/MyDrive/A-Textbook-of-Engineeri

---

**Generate Embeddings**

---

In [18]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

texts = [chunk['text'] for chunk in all_chunks]

embeddings = embedding_model.encode(
    texts,
    convert_to_numpy = True,
    show_progress_bar=True
)

embeddings = np.asarray(embeddings,dtype='float32')

print("Embeddings shape : ",embeddings.shape)

Batches:   0%|          | 0/282 [00:00<?, ?it/s]

Embeddings shape :  (8996, 384)


---

**Create the FAISS Index**

---

In [20]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(embeddings)

print("Vectors in FAISS : ",index.ntotal)

Vectors in FAISS :  8996


---

**Create the Retrieval Function**

---

In [24]:
def retrieve(query,k=3):
  if index.ntotal == 0:
    return []

  query_embedding = embedding_model.encode(
      [query],
      convert_to_numpy = True
  )

  query_embedding = np.asarray(
      query_embedding,
      dtype='float32'
  )
  k = min(k,index.ntotal)

  distances,indices = index.search(query_embedding,k)

  results = []

  for rank,idx in enumerate(indices[0]):
    if idx<0:
      continue

    results.append({
        "rank": rank+1,
        "text": all_chunks[idx]["text"],
        "source": all_chunks[idx]["source"],
        "page": all_chunks[idx]["page"],
        "distance": distances[0][rank]
    })

  return results

---

**Inspect Retrieved Chunks**

---

In [25]:
query = "What is probability?"

results = retrieve(query, k=3)

for result in results:
    print("Rank:", result["rank"])
    print("Distance:", round(result["distance"], 4))
    print("Source:", result["source"])
    print("Page:", result["page"])
    print("Text:", result["text"])
    print("-" * 60)

Rank: 1
Distance: 1.1629
Source: /content/drive/MyDrive/A-Textbook-of-EngineeringP-hysics-by-M-N-Avadhanulu-and-PG-Kshirsagar.pdf
Page: 619
Text: the atom must change by one unit to compensate for the angular momentum carried by the emitted or absorbed photon. 20.26 THE TRANSITION FROM DETERMINISTIC TO PROBABILISTIC NATURE The probabilistic interpretation is a fundamental feature of quantum mechanics. Every particle has a wave function associated with it. The wave function determines the corpuscular characteristics like position, momentum etc in a statistical sense. For instance a particle with a wave function y(x, t) has a probability proportional to | y(x, t)|2 for being found in the neighbourhood of the point x. Nature does not permit any more precise specification of the
------------------------------------------------------------
Rank: 2
Distance: 1.1841
Source: /content/drive/MyDrive/A-Textbook-of-EngineeringP-hysics-by-M-N-Avadhanulu-and-PG-Kshirsagar.pdf
Page: 569
Text: Quantum

---

**Multi-PDF RAG Function**

---

In [26]:
def multi_pdf_rag(query, k=3):

    # 1. Retrieve relevant chunks
    results = retrieve(query, k=k)

    if not results:
        return "No readable PDF content is available.", []

    # 2. Build context with source details
    context = "\n\n".join(
        f"[Source: {r['source']}, Page: {r['page']}]\n{r['text']}"
        for r in results
    )

    # 3. Create prompt
    prompt = f"""
You are a document question-answering assistant.

Answer the question using only the supplied context.
Do not use outside knowledge.

If the context does not contain the answer, say:
"I don't know based on the provided documents."

When possible, mention the source PDF and page number
that support the answer.

Context:
{context}

Question:
{query}

Answer:
"""

    # 4. Generate answer using Groq
    response = model.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0
    )

    answer = response.choices[0].message.content

    return answer, results